# A Small C# Database Workflow



```{contents}
:local:
:depth: 2
```


A C# program usually talks to a database through a provider library. The workflow is the same across many providers: open a connection, create a command, add parameters, execute it, and map results back into C# values.


```{index} database provider
```

## Providers and Connection Strings


A **provider** is a .NET library that knows how to talk to a specific database system. For a small local database, SQLite is a common teaching choice. For deployed systems, you might use SQL Server, PostgreSQL, MySQL, or another service.

A **connection string** tells the provider where the database lives and how to connect.

```csharp
string connectionString = "Data Source=books.db";
```

Do not put real production passwords directly in source code. Store secrets in configuration or environment variables.


```{index} schema creation; database
```

## Creating a Table


The exact package setup depends on the provider, but the application pattern looks like this:

```csharp
using Microsoft.Data.Sqlite;

using var connection = new SqliteConnection("Data Source=books.db");
connection.Open();

using var create = connection.CreateCommand();
create.CommandText = """
CREATE TABLE IF NOT EXISTS Books (
    BookId INTEGER PRIMARY KEY,
    Title TEXT NOT NULL,
    Author TEXT NOT NULL,
    Price REAL NOT NULL
);
""";
create.ExecuteNonQuery();
```

`ExecuteNonQuery` runs a command that does not return result rows, such as `CREATE TABLE`, `INSERT`, `UPDATE`, or `DELETE`.


```{index} Inserting with Parameters
```

## Inserting with Parameters


A parameterized insert keeps C# values separate from SQL text.

```csharp
using var insert = connection.CreateCommand();
insert.CommandText = """
INSERT INTO Books (Title, Author, Price)
VALUES (@title, @author, @price);
""";

insert.Parameters.AddWithValue("@title", "The Pragmatic Programmer");
insert.Parameters.AddWithValue("@author", "David Thomas and Andrew Hunt");
insert.Parameters.AddWithValue("@price", 42.00);
insert.ExecuteNonQuery();
```


```{index} data reader; C#
```

## Reading Results


A query returns rows. A data reader moves through those rows one at a time.

```csharp
record Book(int BookId, string Title, string Author, double Price);

using var query = connection.CreateCommand();
query.CommandText = """
SELECT BookId, Title, Author, Price
FROM Books
WHERE Price <= @maxPrice
ORDER BY Title;
""";
query.Parameters.AddWithValue("@maxPrice", 50.00);

using var reader = query.ExecuteReader();
var books = new List<Book>();

while (reader.Read())
{
    books.Add(new Book(
        reader.GetInt32(0),
        reader.GetString(1),
        reader.GetString(2),
        reader.GetDouble(3)));
}

foreach (Book book in books)
{
    Console.WriteLine($"{book.Title} by {book.Author}: {book.Price:C}");
}
```

The `Book` record is not the database row itself. It is a C# value that your program creates from the row.


```{index} The Workflow
```

## The Workflow


Most small database programs follow this shape:

1. Design the tables and keys.
2. Open a connection.
3. Create tables if needed.
4. Use parameterized commands for inserts, updates, deletes, and searches.
5. Map query results into C# objects or records.
6. Close or dispose the connection.

Later frameworks such as Entity Framework Core automate more of this mapping, but the underlying ideas are the same: tables store data, SQL describes the work, and C# code sends commands and reads results.


## Practice


Sketch a tiny database workflow for one feature in your semester project:

1. name one table;
2. list three columns and a primary key;
3. write one parameterized `INSERT`; and
4. write one `SELECT` that your C# code would use.


```{rubric} Footnotes
```
[^1]: `Microsoft.Data.Sqlite` is a common .NET SQLite provider. A local project may need to add the package before compiling these examples.
